# 第03课：线性变换 —— 矩阵在「做什么」

> **前置要求**：完成第01课（向量）和第02课（矩阵乘法）

## 这节课你将理解

- 上节课学了矩阵乘法怎么算，这节课回答：**它在做什么？**
- 什么是「基向量」？为什么要换一组基？
- AI 里的「嵌入空间」到底是什么意思？
- 为什么 GPT 要把词向量反复「变换」来「变换」去？

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys; sys.path.append('..')
from utils import zh_font

print('准备好了！')

---
# 一、矩阵乘法的几何意义

上节课最后我们看到：用一个矩阵乘一批点，这些点就被旋转了。

其实不止旋转，矩阵可以做很多种变换：

| 变换 | 效果 | AI 里的例子 |
|------|------|------------|
| 旋转 | 转个角度 | 调整特征方向 |
| 缩放 | 拉长/压扁 | 放大重要特征、缩小噪声 |
| 剪切 | 歪一下 | 混合不同特征 |
| 投影 | 高维→低维（信息会丢失） | 降维、注意力输出 |

这些变换有个共同的名字：**线性变换**。

> 💡 **线性变换 = 用矩阵乘法做的变换。**
> 
> 「线性」的意思是：变换前的直线，变换后还是直线（不会弯）。

In [ ]:
# 亲眼看看不同矩阵对一组点的效果

def plot_transform(ax, M, title):
    """画出矩阵 M 对一个正方形的变换效果"""
    # 正方形的4个角 + 回到起点
    square = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]], dtype=float)
    
    # 对每个点做变换：新点 = 旧点 @ M转置
    transformed = square @ M.T
    
    ax.plot(square[:,0], square[:,1], 'b-o', label='原始', markersize=5)
    ax.plot(transformed[:,0], transformed[:,1], 'r-s', label='变换后', markersize=5)
    ax.set_xlim(-2, 3); ax.set_ylim(-2, 3)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(title, fontproperties=zh_font, fontsize=12)
    ax.legend(prop=zh_font if zh_font else None, fontsize=9)


fig, axes = plt.subplots(2, 2, figsize=(10, 10))

# 1. 缩放：x方向拉长2倍
plot_transform(axes[0,0], np.array([[2, 0], [0, 1]]), '缩放（x方向×2）')

# 2. 旋转：转45度
t = np.pi / 4
plot_transform(axes[0,1], np.array([[np.cos(t), -np.sin(t)],
                                     [np.sin(t),  np.cos(t)]]), '旋转45°')

# 3. 剪切：x方向歪
plot_transform(axes[1,0], np.array([[1, 0.5], [0, 1]]), '剪切')

# 4. 投影：压到x轴上（y信息丢失）
plot_transform(axes[1,1], np.array([[1, 0], [0, 0]]), '投影到x轴（y丢失）')

plt.tight_layout()
plt.show()

print('蓝色 = 原始正方形，红色 = 变换后')
print('注意：变换后直线还是直线 → 这就是「线性」的含义')

---
# 二、基向量 —— 坐标系的「尺子」

## 2.1 什么是基向量？

当我们说一个点的坐标是 `[3, 2]`，其实隐含了一个意思：

```
这个点 = 3 × 「往右一步」 + 2 × 「往上一步」
```

这里的「往右一步」和「往上一步」就是**基向量**：

```
e₁ = [1, 0]   （右）
e₂ = [0, 1]   （上）
```

**任何点都可以用基向量「拼」出来**：
```
[3, 2] = 3 × [1, 0] + 2 × [0, 1]
```

> 💡 **基向量 = 坐标系的单位尺子。** 你选不同的尺子，同一个点的坐标就不同。

In [ ]:
# 可视化：用基向量拼出一个点

fig, ax = plt.subplots(figsize=(6, 6))

# 基向量
e1 = np.array([1, 0])
e2 = np.array([0, 1])

# 目标点
point = np.array([3, 2])

# 画基向量（粗箭头）
ax.annotate('', xy=e1, xytext=(0,0), arrowprops=dict(arrowstyle='->', lw=3, color='blue'))
ax.text(0.5, -0.2, 'e₁=[1,0]', fontsize=11, color='blue', ha='center')

ax.annotate('', xy=e2, xytext=(0,0), arrowprops=dict(arrowstyle='->', lw=3, color='green'))
ax.text(-0.3, 0.5, 'e₂=[0,1]', fontsize=11, color='green', ha='center')

# 画「拼」的过程：先走3步e1，再走2步e2
ax.annotate('', xy=(3, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='blue', linestyle='--'))
ax.annotate('', xy=(3, 2), xytext=(3, 0),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='green', linestyle='--'))

ax.plot(*point, 'ro', markersize=12)
ax.text(3.15, 2.15, '[3, 2]', fontsize=13, color='red')

ax.set_xlim(-0.8, 4); ax.set_ylim(-0.8, 3)
ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_title('[3,2] = 3×e₁ + 2×e₂', fontsize=14)
plt.show()

# 代码验证
result = 3 * e1 + 2 * e2
print(f'3×{e1} + 2×{e2} = {result}')

## 2.2 线性变换 = 移动基向量

这是本课**最关键的直觉**：

> 🔑 **要理解一个矩阵「做了什么变换」，只需要看它把基向量变到了哪里。**

矩阵的第 1 列 = e₁ 被变到了哪里。

矩阵的第 2 列 = e₂ 被变到了哪里。

```
矩阵 M = [[a, b],    意思是：
           [c, d]]       e₁=[1,0] → [a, c]  （第1列）
                         e₂=[0,1] → [b, d]  （第2列）
```

知道了新的基向量，所有其他点的新位置就自动确定了（因为是「线性」的）。

In [ ]:
# 验证：矩阵的列 = 变换后的基向量

M = np.array([[2, -1],
              [1,  1]])

e1 = np.array([1, 0])
e2 = np.array([0, 1])

new_e1 = M @ e1   # e1 变换后去了哪里？
new_e2 = M @ e2   # e2 变换后去了哪里？

print(f'M 的第1列: {M[:, 0]}')
print(f'e1 变换后: {new_e1}')   # 跟 M 的第1列一样！
print()
print(f'M 的第2列: {M[:, 1]}')
print(f'e2 变换后: {new_e2}')   # 跟 M 的第2列一样！

In [ ]:
# 可视化：看基向量怎么被「搬家」的

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, title, basis in [
    (axes[0], '变换前', [e1, e2]),
    (axes[1], '变换后', [new_e1, new_e2]),
]:
    colors = ['blue', 'green']
    labels = ['e₁', 'e₂']
    for vec, c, lab in zip(basis, colors, labels):
        ax.annotate('', xy=vec, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', lw=3, color=c))
        ax.text(vec[0]*1.1, vec[1]*1.1, f'{lab}={list(vec)}',
                fontsize=11, color=c)
    ax.set_xlim(-2, 3); ax.set_ylim(-1.5, 2)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(title, fontsize=14, fontproperties=zh_font)

plt.tight_layout()
plt.show()

print('矩阵 M 做了什么？')
print(f'  e₁ 从 [1,0] 搬到了 {list(new_e1)}')
print(f'  e₂ 从 [0,1] 搬到了 {list(new_e2)}')
print('其他所有点跟着一起动 —— 这就是「线性变换」的全部。')

---
# 三、基变换 —— 换一副「眼镜」看世界

## 3.1 同一个点，不同的坐标

想象你站在一个房间里，面朝北。你说「桌子在右边 3 米、前方 2 米」。

现在你转了 90°，面朝东。同一张桌子，你会说「前方 3 米、左边 2 米」。

**桌子没动，但你的描述变了 —— 因为你的「坐标系」变了。**

这就是**基变换**：同一组数据，换一组基向量来描述。

## 3.2 为什么 AI 要做基变换？

原始坐标系可能不好用。换一个聪明的坐标系，**数据的规律会变得更明显**。

举个例子：

| 原始特征 | 更好的特征 |
|---------|----------|
| 身高(cm)、体重(kg) | BMI指数、体型分类 |
| 像素亮度 | 边缘、纹理、形状 |
| 单词ID | 语义向量（相似词距离近） |

**AI 的训练过程，本质上就是在学习一组最好的「基向量」（特征）。**

In [ ]:
# 直观例子：换一组基，让数据变「整齐」

np.random.seed(0)

# 生成沿对角线分布的数据点
# （想象：身高和体重是正相关的，数据点沿对角线散布）
n = 50
t = np.random.randn(n)
noise = np.random.randn(n) * 0.3
data = np.column_stack([t + noise, t * 0.8 - noise])   # (50, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图：原始坐标系
ax = axes[0]
ax.scatter(data[:, 0], data[:, 1], alpha=0.6)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_title('原始坐标系：数据是歪的', fontproperties=zh_font, fontsize=13)

# 找到数据的主方向（后面第6课会详细讲PCA，现在先感受一下）
# 简单理解：找一组新基，让数据沿着新的坐标轴排列
cov = np.cov(data.T)              # 协方差矩阵
eigenvalues, P = np.linalg.eigh(cov)  # 特征向量 = 新的基
P = P[:, ::-1]                    # 大的排前面

# 基变换：data_new = data @ P
data_new = data @ P

# 右图：新坐标系
ax = axes[1]
ax.scatter(data_new[:, 0], data_new[:, 1], alpha=0.6, color='orange')
ax.set_xlabel('新轴1（主方向）'); ax.set_ylabel('新轴2（次方向）')
ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_title('换基后：数据变「正」了', fontproperties=zh_font, fontsize=13)

plt.tight_layout()
plt.show()

print('同样的数据，换了一组基向量，规律一目了然。')
print('这就是 PCA（主成分分析）的核心思想 —— 第6课会详细讲。')

---
# 四、变换的组合 = 矩阵的乘法

如果你想**先旋转，再缩放**，怎么做？

```
方法1（慢）：x → 旋转矩阵R @ x → 缩放矩阵S @ 结果
方法2（快）：先算 M = S @ R，然后 M @ x 一步搞定
```

**矩阵乘法 = 组合变换。** 这就是为什么矩阵乘法满足结合律：`(A @ B) @ C = A @ (B @ C)`

> 💡 GPT 里有 96 层 Transformer（以 GPT-3 为例）。
> 每一层都在对词向量做一次「变换」。
> 96 层叠起来 = 一个超级复杂的变换，能把「原始词向量」变成「理解了上下文语义的向量」。

In [ ]:
# 组合变换：先旋转30°，再x方向缩放2倍

t = np.pi / 6   # 30°
R = np.array([[np.cos(t), -np.sin(t)],
              [np.sin(t),  np.cos(t)]])    # 旋转

S = np.array([[2, 0],
              [0, 1]])                      # 缩放

# 组合矩阵
M = S @ R   # 先 R 再 S（注意：矩阵从右往左读！）

# 画3种情况对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

square = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]], dtype=float)

for ax, mat, title in [
    (axes[0], R, '只旋转30°'),
    (axes[1], S, '只缩放x×2'),
    (axes[2], M, '先旋转再缩放（组合）'),
]:
    transformed = square @ mat.T
    ax.plot(square[:,0], square[:,1], 'b-o', markersize=4, label='原始')
    ax.plot(transformed[:,0], transformed[:,1], 'r-s', markersize=4, label='变换后')
    ax.set_xlim(-1, 3); ax.set_ylim(-1, 2.5)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(title, fontproperties=zh_font, fontsize=12)
    ax.legend(prop=zh_font if zh_font else None, fontsize=9)

plt.tight_layout()
plt.show()

print('第3张图 = 第1张 + 第2张 的效果叠加')
print('GPT 96层 = 96个变换叠加 → 超级复杂的变换')

---
# 五、单位矩阵 —— 什么都不做的变换

有一种特殊的矩阵叫**单位矩阵**（Identity Matrix），写作 **I**。

它的效果是：**什么都不变**。（就像乘以 1 一样。）

```
I = [[1, 0],
     [0, 1]]

I @ 任何向量 = 原来那个向量
```

对角线全是 1，其他全是 0。

**AI 中的用途**：残差连接（Residual Connection）。
```
输出 = 输入 + 变换(输入)
     = I×输入 + 变换(输入)
```
GPT 每一层都有残差连接 —— 保证即使变换搞砸了，至少原始信息还在。

In [ ]:
# 单位矩阵
I = np.eye(3)   # eye = I = 单位矩阵
print('3×3 单位矩阵:')
print(I)

v = np.array([5, -3, 7])
print(f'\nI @ {v} = {I @ v}')   # 不变！

# 模拟残差连接
x = np.array([1.0, 2.0, 3.0])      # 输入
W = np.random.randn(3, 3) * 0.1     # 一个很小的变换
transform = x @ W                    # 变换结果
output = x + transform               # 残差连接：原始 + 变换

print(f'\n残差连接: {np.round(x, 2)} + {np.round(transform, 2)} = {np.round(output, 2)}')
print('即使变换很小，原始信息也保留了 → 训练更稳定')

---
# 六、在 AI 中的全景图

现在把这三节课学的东西串起来，看 GPT 处理一句话时发生了什么：

```
输入："猫 吃 鱼"

第1步 - 词嵌入（第1课）:
    "猫" → [0.2, 0.8, ...]   把词变成向量
    "吃" → [0.1, 0.5, ...]
    "鱼" → [0.3, 0.9, ...]

第2步 - 线性变换（第2、3课）:
    每一层 Transformer 对这些向量做矩阵乘法
    = 在不同的「坐标系」（基）下重新理解这些词
    
    第1层可能学到：猫是名词、鱼是名词、吃是动词
    第5层可能学到：猫是吃的主语、鱼是吃的宾语
    第50层可能学到：整句话在说「某种动物的饮食行为」

第3步 - 输出:
    最后一层的向量包含了对整句话的「理解」
    再用一次矩阵乘法，从向量变回词 → 预测下一个词
```

**每一层 = 一次线性变换 = 换一种「视角」理解同一组词。**

In [ ]:
# 模拟：3层变换，观察向量怎么被逐层改变

np.random.seed(42)

# 3个词的初始向量（4维）
words = {'猫': np.array([0.2, 0.8, -0.1, 0.5]),
         '吃': np.array([0.1, 0.5, 0.3, -0.2]),
         '鱼': np.array([0.3, 0.9, 0.1, 0.4])}

# 3层变换矩阵
layers = [np.random.randn(4, 4) * 0.5 for _ in range(3)]

print('初始词向量（简化4维）:')
for w, v in words.items():
    print(f'  {w}: {np.round(v, 2)}')

# 逐层变换
current = words.copy()
for i, W in enumerate(layers):
    print(f'\n--- 第{i+1}层变换后 ---')
    new_current = {}
    for w, v in current.items():
        new_v = v @ W + v   # 线性变换 + 残差连接
        new_current[w] = new_v
        print(f'  {w}: {np.round(new_v, 2)}')
    current = new_current

print('\n每一层都在改变词向量 → 注入新的「理解」。')

---
# 七、本节总结

| 概念 | 一句话理解 | AI 用途 |
|-----|----------|--------|
| **线性变换** | 用矩阵乘法对向量做旋转/缩放/投影 | 神经网络每一层 |
| **基向量** | 坐标系的单位尺子 | 特征空间的坐标轴 |
| **矩阵的列** | 变换后的基向量 | 理解矩阵在做什么 |
| **基变换** | 换一组基描述同样的数据 | 找更好的特征表示 |
| **组合变换** | 多个矩阵连乘 | GPT 多层叠加 |
| **单位矩阵** | 什么都不做的变换 | 残差连接 |

---

# 八、练习

## 题1：看矩阵猜变换

下面这个矩阵把 e₁ 和 e₂ 分别变到了哪里？它做的是什么变换？

In [ ]:
M = np.array([[0, -1],
              [1,  0]])

# TODO: 算 M @ [1,0] 和 M @ [0,1]，猜猜这是什么变换
# 提示：[1,0] 变到了 [0,1]... 是不是像旋转？旋转了多少度？


## 题2：组合两个变换

先用上面的 M（旋转90°），再用 `S = [[2,0],[0,2]]`（放大2倍）。

算出组合矩阵，并验证效果。

In [ ]:
# TODO: 算 S @ M，然后用它变换 [1, 0] 看看结果


## 题3：残差连接

创建一个随机 3×3 矩阵 W，对向量 `x = [1, 2, 3]` 做变换。

分别算「不带残差」和「带残差」的结果，比较哪个跟原始 x 更像。

In [ ]:
# TODO
# no_residual = x @ W
# with_residual = x + x @ W
# 用余弦相似度比较哪个跟 x 更像


---
## 答案

In [ ]:
# === 题1 ===
M = np.array([[0, -1], [1, 0]])
print('题1:')
print(f'  e₁ → {M @ np.array([1, 0])}')  # [0, 1]
print(f'  e₂ → {M @ np.array([0, 1])}')  # [-1, 0]
print('  [1,0]→[0,1], [0,1]→[-1,0] → 逆时针旋转90°')

# === 题2 ===
S = np.array([[2, 0], [0, 2]])
combo = S @ M   # 先旋转再放大
print(f'\n题2: 组合矩阵 = \n{combo}')
print(f'  [1,0] → {combo @ np.array([1, 0])}')  # [0, 2]：旋转90°再放大2倍

# === 题3 ===
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

np.random.seed(7)
x = np.array([1.0, 2.0, 3.0])
W = np.random.randn(3, 3) * 0.5

no_res = x @ W
with_res = x + x @ W

print(f'\n题3:')
print(f'  不带残差 cos(x, xW)   = {cosine_sim(x, no_res):.3f}')
print(f'  带残差   cos(x, x+xW) = {cosine_sim(x, with_res):.3f}')
print('  带残差的结果跟原始x更像 → 信息保留更多')

---
🎉 **第03课完成！**

下一课（第04课）：范数、内积、正交 —— 注意力机制和归一化的数学基础。